In [ ]:
import pandas as pd
import zipfile
import os

In [ ]:
import pandas as pd
import zipfile
import os
import time
import random
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException

# Configurações do Chrome
def setup_driver():
    chrome_options = Options()
    chrome_options.add_argument('--headless')  # Remove para ver o navegador
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument('--window-size=1920,1080')
    chrome_options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36')
    
    # Adiciona outras opções para evitar detecção
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    chrome_options.add_argument('--disable-blink-features=AutomationControlled')
    
    driver = webdriver.Chrome(options=chrome_options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    
    return driver

def fetch_table_with_selenium(driver, url, table_id):
    """Usa Selenium para extrair tabelas do fbref.com"""
    try:
        print(f"Acessando: {url}")
        driver.get(url)
        
        # Espera a página carregar
        time.sleep(random.uniform(3, 6))
        
        # Tenta encontrar a tabela
        try:
            # Espera até que a tabela esteja presente
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.ID, table_id))
            )
            
            # Encontra a tabela pelo ID
            table_element = driver.find_element(By.ID, table_id)
            html_content = table_element.get_attribute('outerHTML')
            
            # Extrai a tabela usando pandas
            df = pd.read_html(html_content, header=[1])[0]
            print(f"✓ Tabela {table_id} extraída com sucesso!")
            return df
            
        except TimeoutException:
            print(f"✗ Timeout: Tabela {table_id} não encontrada em {url}")
            return None
        except NoSuchElementException:
            print(f"✗ Tabela {table_id} não encontrada em {url}")
            return None
            
    except Exception as e:
        print(f"✗ Erro ao acessar {url}: {e}")
        return None

def salvar_tabelas_zip(nome_zip, formato="csv", **tabelas):
    """Salva as tabelas em um arquivo ZIP"""
    pasta_temp = "_temp_tabelas"
    os.makedirs(pasta_temp, exist_ok=True)
    
    caminhos_arquivos = []
    for nome, df in tabelas.items():
        if df is not None and not df.empty:
            caminho = os.path.join(pasta_temp, f"{nome}.{formato}")
            if formato == "csv":
                df.to_csv(caminho, index=False, encoding='utf-8-sig')
            elif formato == "xlsx":
                df.to_excel(caminho, index=False)
            else:
                raise ValueError("Formato inválido: use 'csv' ou 'xlsx'")
            caminhos_arquivos.append(caminho)
            print(f"✓ {nome} salvo com {len(df)} linhas")
    
    if not caminhos_arquivos:
        print("✗ Nenhuma tabela válida para salvar")
        return None
    
    zip_path = f"{nome_zip}.zip"
    with zipfile.ZipFile(zip_path, "w") as zipf:
        for arquivo in caminhos_arquivos:
            zipf.write(arquivo, os.path.basename(arquivo))
    
    # Limpeza
    for arquivo in caminhos_arquivos:
        os.remove(arquivo)
    os.rmdir(pasta_temp)
    
    print(f"✅ Arquivo ZIP criado: {zip_path}")
    return zip_path

def main():
    # URLs e IDs das tabelas para a temporada atual
    urls_temporada_atual = {
        'standard_df': ('https://fbref.com/en/comps/Big5/stats/players/Big-5-European-Leagues-Stats', 'stats_standard'),
        'goalkeeping_df': ('https://fbref.com/en/comps/Big5/keepers/players/Big-5-European-Leagues-Stats', 'stats_keeper'),
        'adv_goalkeeping_df': ('https://fbref.com/en/comps/Big5/keepersadv/players/Big-5-European-Leagues-Stats', 'stats_keeper_adv'),
        'shooting_df': ('https://fbref.com/en/comps/Big5/shooting/players/Big-5-European-Leagues-Stats', 'stats_shooting'),
        'passing_df': ('https://fbref.com/en/comps/Big5/passing/players/Big-5-European-Leagues-Stats', 'stats_passing'),
        'pass_types_df': ('https://fbref.com/en/comps/Big5/passing_types/players/Big-5-European-Leagues-Stats', 'stats_passing_types'),
        'gca_df': ('https://fbref.com/en/comps/Big5/gca/players/Big-5-European-Leagues-Stats', 'stats_gca'),
        'defense_df': ('https://fbref.com/en/comps/Big5/defense/players/Big-5-European-Leagues-Stats', 'stats_defense'),
        'possession_df': ('https://fbref.com/en/comps/Big5/possession/players/Big-5-European-Leagues-Stats', 'stats_possession'),
        'playing_time_df': ('https://fbref.com/en/comps/Big5/playingtime/players/Big-5-European-Leagues-Stats', 'stats_playing_time'),
        'misc_df': ('https://fbref.com/en/comps/Big5/misc/players/Big-5-European-Leagues-Stats', 'stats_misc')
    }

    # URLs para temporada 2022-2023 (alternativa)
    urls_2022_2023 = {
        'standard_df': ('https://fbref.com/en/comps/Big5/2020-2021/stats/players/2020-2021-Big-5-European-Leagues-Stats', 'stats_standard'),
        'goalkeeping_df': ('https://fbref.com/en/comps/Big5/2020-2021/keepers/players/2020-2021-Big-5-European-Leagues-Stats', 'stats_keeper'),
        'adv_goalkeeping_df': ('https://fbref.com/en/comps/Big5/2020-2021/keepersadv/players/2020-2021-Big-5-European-Leagues-Stats', 'stats_keeper_adv'),
        'shooting_df': ('https://fbref.com/en/comps/Big5/2020-2021/shooting/players/2020-2021-Big-5-European-Leagues-Stats', 'stats_shooting'),
        'passing_df': ('https://fbref.com/en/comps/Big5/2020-2021/passing/players/2020-2021-Big-5-European-Leagues-Stats', 'stats_passing'),
        'pass_types_df': ('https://fbref.com/en/comps/Big5/2020-2021/passing_types/players/2020-2021-Big-5-European-Leagues-Stats', 'stats_passing_types'),
        'gca_df': ('https://fbref.com/en/comps/Big5/2020-2021/gca/players/2020-2021-Big-5-European-Leagues-Stats', 'stats_gca'),
        'defense_df': ('https://fbref.com/en/comps/Big5/2020-2021/defense/players/2020-2021-Big-5-European-Leagues-Stats', 'stats_defense'),
        'possession_df': ('https://fbref.com/en/comps/Big5/2020-2021/possession/players/2020-2021-Big-5-European-Leagues-Stats', 'stats_possession'),
        'playing_time_df': ('https://fbref.com/en/comps/Big5/2020-2021/playingtime/players/2020-2021-Big-5-European-Leagues-Stats', 'stats_playing_time'),
        'misc_df': ('https://fbref.com/en/comps/Big5/2020-2021/misc/players/2020-2021-Big-5-European-Leagues-Stats', 'stats_misc')
    }

    # Escolha qual conjunto de URLs usar
    urls_para_usar = urls_2022_2023  # ou urls_2022_2023

    print("🚀 Iniciando extração de dados do fbref.com...")
    
    # Configura o driver
    driver = setup_driver()
    
    try:
        tabelas = {}
        
        for nome, (url, table_id) in urls_para_usar.items():
            df = fetch_table_with_selenium(driver, url, table_id)
            if df is not None:
                tabelas[nome] = df
            
            # Delay aleatório entre requisições para evitar bloqueio
            delay = random.uniform(5, 10)
            print(f"⏳ Aguardando {delay:.1f} segundos...")
            time.sleep(delay)
        
        # Fecha o driver
        driver.quit()
        
        # Salva as tabelas extraídas
        if tabelas:
            salvar_tabelas_zip("./estatisticas/estatisticas_fbref", formato="csv", **tabelas)
        else:
            print("✗ Nenhuma tabela foi extraída com sucesso")
            
    except Exception as e:
        print(f"✗ Erro durante a execução: {e}")
        driver.quit()

if __name__ == "__main__":
    main()

🚀 Iniciando extração de dados do fbref.com...
Acessando: https://fbref.com/en/comps/Big5/2020-2021/stats/players/2020-2021-Big-5-European-Leagues-Stats


C:\Users\guilh\AppData\Local\Temp\ipykernel_4228\288704899.py:54: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html_content, header=[1])[0]


✓ Tabela stats_standard extraída com sucesso!
⏳ Aguardando 10.0 segundos...
Acessando: https://fbref.com/en/comps/Big5/2020-2021/keepers/players/2020-2021-Big-5-European-Leagues-Stats


C:\Users\guilh\AppData\Local\Temp\ipykernel_4228\288704899.py:54: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html_content, header=[1])[0]


✓ Tabela stats_keeper extraída com sucesso!
⏳ Aguardando 7.8 segundos...
Acessando: https://fbref.com/en/comps/Big5/2020-2021/keepersadv/players/2020-2021-Big-5-European-Leagues-Stats


C:\Users\guilh\AppData\Local\Temp\ipykernel_4228\288704899.py:54: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html_content, header=[1])[0]


... (13 outputs left out)

C:\Users\guilh\AppData\Local\Temp\ipykernel_4228\288704899.py:54: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html_content, header=[1])[0]


✓ Tabela stats_playing_time extraída com sucesso!
⏳ Aguardando 9.7 segundos...
Acessando: https://fbref.com/en/comps/Big5/2020-2021/misc/players/2020-2021-Big-5-European-Leagues-Stats


C:\Users\guilh\AppData\Local\Temp\ipykernel_4228\288704899.py:54: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(html_content, header=[1])[0]


✓ Tabela stats_misc extraída com sucesso!
⏳ Aguardando 5.2 segundos...
✓ standard_df salvo com 2934 linhas
✓ goalkeeping_df salvo com 216 linhas
✓ adv_goalkeeping_df salvo com 216 linhas
✓ shooting_df salvo com 2934 linhas
✓ passing_df salvo com 2934 linhas
✓ pass_types_df salvo com 2934 linhas
✓ gca_df salvo com 2934 linhas
✓ defense_df salvo com 2934 linhas
✓ possession_df salvo com 2934 linhas
✓ playing_time_df salvo com 3580 linhas
✓ misc_df salvo com 2934 linhas
✅ Arquivo ZIP criado: estatisticas_fbref.zip


In [ ]:
def salvar_tabelas_zip(nome_zip, formato="csv", **tabelas):
    pasta_temp = "_temp_tabelas"
    os.makedirs(pasta_temp, exist_ok=True)
    
    caminhos_arquivos = []
    for nome, df in tabelas.items():
        caminho = os.path.join(pasta_temp, f"{nome}.{formato}")
        if formato == "csv":
            df.to_csv(caminho, index=False)
        elif formato == "xlsx":
            df.to_excel(caminho, index=False)
        else:
            raise ValueError("Formato inválido: use 'csv' ou 'xlsx'")
        caminhos_arquivos.append(caminho)
    
    zip_path = f"{nome_zip}.zip"
    with zipfile.ZipFile(zip_path, "w") as zipf:
        for arquivo in caminhos_arquivos:
            zipf.write(arquivo, os.path.basename(arquivo))
    
    for arquivo in caminhos_arquivos:
        os.remove(arquivo)
    os.rmdir(pasta_temp)
    
    print(f"Arquivo ZIP criado: {zip_path}")
    return zip_path

In [ ]:
# TEMPORADA ATUAL
# Player Standard Stats
standard_df = pd.read_html('https://fbref.com/en/comps/Big5/stats/players/Big-5-European-Leagues-Stats', attrs={'id':"stats_standard"}, header=[1])[0]
# Player Goalkeeping Stats
goalkeeping_df = pd.read_html('https://fbref.com/en/comps/Big5/keepers/players/Big-5-European-Leagues-Stats', attrs={'id':"stats_keeper"}, header=[1])[0]
# Player Goalkeeping Advanced Stats
adv_goalkeeping_df = pd.read_html('https://fbref.com/en/comps/Big5/keepersadv/players/Big-5-European-Leagues-Stats', attrs={'id':"stats_keeper_adv"}, header=[1])[0]
# Player Shooting Stats
shooting_df = pd.read_html('https://fbref.com/en/comps/Big5/shooting/players/Big-5-European-Leagues-Stats', attrs={'id':"stats_shooting"}, header=[1])[0]
# Player Passing Stats
passing_df = pd.read_html('https://fbref.com/en/comps/Big5/passing/players/Big-5-European-Leagues-Stats', attrs={'id':"stats_passing"}, header=[1])[0]
# Player Pass Types Stats
pass_types_df = pd.read_html('https://fbref.com/en/comps/Big5/passing_types/players/Big-5-European-Leagues-Stats', attrs={'id':"stats_passing_types"}, header=[1])[0]
# Player Goal and Shot Creation Stats
gca_df = pd.read_html('https://fbref.com/en/comps/Big5/gca/players/Big-5-European-Leagues-Stats', attrs={'id':"stats_gca"}, header=[1])[0]
# Player Defensive Actions Stats
defense_df = pd.read_html('https://fbref.com/en/comps/Big5/defense/players/Big-5-European-Leagues-Stats', attrs={'id':"stats_defense"}, header=[1])[0]
# Player Possession Stats
possession_df = pd.read_html('https://fbref.com/en/comps/Big5/possession/players/Big-5-European-Leagues-Stats', attrs={'id':"stats_possession"}, header=[1])[0]
# Player Playing Time Stats
playing_time_df = pd.read_html('https://fbref.com/en/comps/Big5/playingtime/players/Big-5-European-Leagues-Stats', attrs={'id':"stats_playing_time"}, header=[1])[0]
# Player Miscellaneous Stats
misc_df = pd.read_html('https://fbref.com/en/comps/Big5/misc/players/Big-5-European-Leagues-Stats', attrs={'id':"stats_misc"}, header=[1])[0]

In [ ]:
# TEMPORADAS ANTERIORES
# Player Standard Stats
standard_df = pd.read_html('https://fbref.com/en/comps/Big5/2022-2023/stats/players/2022-2023-Big-5-European-Leagues-Stats', attrs={'id':"stats_standard"}, header=[1])[0]
# Player Goalkeeping Stats
goalkeeping_df = pd.read_html('https://fbref.com/en/comps/Big5/2022-2023/keepers/players/2022-2023-Big-5-European-Leagues-Stats', attrs={'id':"stats_keeper"}, header=[1])[0]
# Player Goalkeeping Advanced Stats
adv_goalkeeping_df = pd.read_html('https://fbref.com/en/comps/Big5/2022-2023/keepersadv/players/2022-2023-Big-5-European-Leagues-Stats', attrs={'id':"stats_keeper_adv"}, header=[1])[0]
# Player Shooting Stats
shooting_df = pd.read_html('https://fbref.com/en/comps/Big5/2023-2024/shooting/players/2023-2024-Big-5-European-Leagues-Stats', attrs={'id':"stats_shooting"}, header=[1])[0]
# Player Passing Stats
passing_df = pd.read_html('https://fbref.com/en/comps/Big5/2023-2024/passing/players/2023-2024-Big-5-European-Leagues-Stats', attrs={'id':"stats_passing"}, header=[1])[0]
# Player Pass Types Stats
pass_types_df = pd.read_html('https://fbref.com/en/comps/Big5/2023-2024/passing_types/players/2023-2024-Big-5-European-Leagues-Stats', attrs={'id':"stats_passing_types"}, header=[1])[0]
# Player Goal and Shot Creation Stats
gca_df = pd.read_html('https://fbref.com/en/comps/Big5/2023-2024/gca/players/2023-2024-Big-5-European-Leagues-Stats', attrs={'id':"stats_gca"}, header=[1])[0]
# Player Defensive Actions Stats
defense_df = pd.read_html('https://fbref.com/en/comps/Big5/2023-2024/defense/players/2023-2024-Big-5-European-Leagues-Stats', attrs={'id':"stats_defense"}, header=[1])[0]
# Player Possession Stats
possession_df = pd.read_html('https://fbref.com/en/comps/Big5/2023-2024/possession/players/2023-2024-Big-5-European-Leagues-Stats', attrs={'id':"stats_possession"}, header=[1])[0]
# Player Playing Time Stats
playing_time_df = pd.read_html('https://fbref.com/en/comps/Big5/2023-2024/playingtime/players/2023-2024-Big-5-European-Leagues-Stats', attrs={'id':"stats_playing_time"}, header=[1])[0]
# Player Miscellaneous Stats
misc_df = pd.read_html('https://fbref.com/en/comps/Big5/2023-2024/misc/players/2023-2024-Big-5-European-Leagues-Stats', attrs={'id':"stats_misc"}, header=[1])[0]

In [ ]:
salvar_tabelas_zip(
    "estatisticas_22-23",
    formato="csv",
    standard_df=standard_df,
    goalkeeping_df=goalkeeping_df,
    adv_goalkeeping_df=adv_goalkeeping_df,
    shooting_df=shooting_df,
    passing_df=passing_df,
    pass_types_df=pass_types_df,
    gca_df=gca_df,
    defense_df=defense_df,
    possession_df=possession_df,
    playing_time_df=playing_time_df,
    misc_df=misc_df
)